# MEMO FIGURES CHECK

In [24]:
import pandas as pd

# --- Load once, filter to Total U.S. / Periods 1-32 (the window the memo compares) ---
d25 = pd.read_csv('2025.csv')
d26 = pd.read_csv('2026PartYear.csv')
us25 = d25[(d25['Geography'] == 'Total U.S.') & (d25['Period'] <= 32)]
us26 = d26[d26['Geography'] == 'Total U.S.']                      # already only has P1-32

pct = lambda a, b: (b / a - 1) * 100                               # simple % change helper
wavg = lambda df: (df['ASP Current Year'] * df['Total Bulk and Bags']).sum() / df['Total Bulk and Bags'].sum()

In [25]:
# --- 1) Volume checks: Total / Conventional / Organic  (✓ = simple % math) ---
for label, mask in [('Total', slice(None)), ('Conventional', 'Conventional'), ('Organic', 'Organic')]:
    v25 = (us25 if mask == slice(None) else us25[us25.Type == mask])['Total Bulk and Bags'].sum()
    v26 = (us26 if mask == slice(None) else us26[us26.Type == mask])['Total Bulk and Bags'].sum()
    print(f"{label:12s} volume: {v25:,.0f} -> {v26:,.0f}  ({pct(v25, v26):+.1f}%) \u2713")

Total        volume: 1,870,698,865 -> 2,125,800,177  (+13.6%) ✓
Conventional volume: 1,764,839,344 -> 2,020,684,608  (+14.5%) ✓
Organic      volume: 105,859,521 -> 105,115,569  (-0.7%) ✓


In [26]:
# --- 2) ASP checks: weighted-average price, both types + overall  (✓ = simple math) ---
for label, mask in [('Total', slice(None)), ('Conventional', 'Conventional'), ('Organic', 'Organic')]:
    a25 = wavg(us25 if mask == slice(None) else us25[us25.Type == mask])
    a26 = wavg(us26 if mask == slice(None) else us26[us26.Type == mask])
    print(f"{label:12s} ASP:    {a25:.3f} -> {a26:.3f}  ({pct(a25, a26):+.1f}%) \u2713")

Total        ASP:    1.245 -> 1.008  (-19.1%) ✓
Conventional ASP:    1.215 -> 0.979  (-19.4%) ✓
Organic      ASP:    1.741 -> 1.562  (-10.3%) ✓


In [27]:
# --- 3) "Every period higher/lower" claims -> need period-by-period compare, not simple math ---
for label, col, cmp in [('Conv volume higher every period', 'Total Bulk and Bags', 'gt'),
                         ('Organic volume up count',        'Total Bulk and Bags', 'gt'),
                         ('Conv ASP lower every period',     'ASP Current Year',    'lt'),
                         ('Organic ASP lower every period',  'ASP Current Year',    'lt')]:
    ty = 'Conventional' if 'Conv' in label else 'Organic'
    s25 = us25[us25.Type == ty].set_index('Period')[col].sort_index()
    s26 = us26[us26.Type == ty].set_index('Period')[col].sort_index()
    n = (s26 > s25).sum() if cmp == 'gt' else (s26 < s25).sum()
    print(f"{label}: {n}/32 periods")

Conv volume higher every period: 32/32 periods
Organic volume up count: 9/32 periods
Conv ASP lower every period: 32/32 periods
Organic ASP lower every period: 32/32 periods


In [28]:
# --- 4) Peak/trough periods for 2026 conventional volume & ASP (single lookups, no re-filtering) ---
cv26 = us26[us26.Type == 'Conventional'].set_index('Period')['Total Bulk and Bags'].sort_index()
ca26 = us26[us26.Type == 'Conventional'].set_index('Period')['ASP Current Year'].sort_index()
print(f"Conv volume max: P{cv26.idxmax()} = {cv26.max():,.0f}; P26 = {cv26[26]:,.0f}; P32 = {cv26[32]:,.0f}")
print(f"Conv ASP min: P{ca26.idxmin()} = {ca26.min():.3f}; ASP max: P{ca26.idxmax()} = {ca26.max():.3f}")

Conv volume max: P6 = 75,968,847; P26 = 54,220,324; P32 = 57,788,036
Conv ASP min: P6 = 0.892; ASP max: P26 = 1.187


In [29]:
# --- 5) Product-mix share checks (✓ = simple % of total math) ---
def shares(df):
    t = df['Total Bulk and Bags'].sum()
    return {k: df[df.Type == 'Conventional']['Total Bulk and Bags'].sum() / t * 100 if k == 'Conventional'
            else df[df.Type == 'Organic']['Total Bulk and Bags'].sum() / t * 100 if k == 'Organic'
            else df[k].sum() / t * 100 for k in ('Conventional', 'Organic', '4046', '4225', '4770', 'TotalBagged')}

for k in ('Conventional', 'Organic', '4046', '4225', '4770', 'TotalBagged'):
    s25, s26 = shares(us25)[k], shares(us26)[k]
    print(f"{k:12s} share: {s25:.1f}% -> {s26:.1f}%  \u2713")

Conventional share: 94.3% -> 95.1%  ✓
Organic      share: 5.7% -> 4.9%  ✓
4046         share: 46.4% -> 41.8%  ✓
4225         share: 10.5% -> 12.6%  ✓
4770         share: 1.3% -> 2.3%  ✓
TotalBagged  share: 41.8% -> 43.3%  ✓
